In [ ]:
"""
Rashomon Epsilon Sweep Experiment (MC Dropout Version)
======================================================
Faithfully implements the Rashomon-restricted posterior P_R(theta) by:
  1. Training a single MC Dropout base model.
  2. Sampling N frozen, deterministic dropout masks to form a hypothesis space.
  3. Computing each mask's NLL on a held-out validation set.
  4. For a given epsilon, keeping only masks whose loss satisfies
         L(theta_i) <= L* + epsilon
     where L* is the best (minimum) mask loss.
  5. Training the CVAE with R_epsilon computed by averaging ONLY over
     the in-set masks, approximating E_{theta ~ P_R(theta)}[log P_theta(...)].
  6. Sweeping epsilon and comparing all metrics against a set of
     baselines.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
import random
import time
import copy
from collections import defaultdict
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Initial setup for bayes preds & variance.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_bayesian_prediction_stats(model, x, target_class, n_samples=100):
    """MC Dropout forward pass — returns (mean_prob, variance, mean_log_prob)."""
    model.train()
    x          = x.to(DEVICE)
    batch_size = x.shape[0]
    x_rep      = x.repeat(n_samples, 1)
    preds      = model(x_rep).view(n_samples, batch_size, -1)
    log_probs      = preds[:, :, target_class]
    mean_log_prob  = log_probs.mean()
    mean_prob      = torch.exp(log_probs).mean()
    variance       = torch.exp(log_probs).var()
    return mean_prob, variance, mean_log_prob


# Data preprocessing - deals with some UCI repo datasets.
class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels   = torch.tensor(labels,   dtype=torch.long)
    def __len__(self):          return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]


def get_dataset(name='credit'):
    print(f"\n--- Loading '{name}' ---")
    if name == 'adult_income':
        ds = fetch_ucirepo(id=2)
        X  = ds.data.features
        y  = ds.data.targets['income'].str.strip().replace({'<=50K': 0, '>50K': 1})
        class_names = ['<=50K', '>50K']
    elif name == 'credit':
        ds = fetch_ucirepo(id=144)
        X  = ds.data.features
        y  = ds.data.targets.iloc[:, 0].map({1: 0, 2: 1})
        class_names = ['Good Credit', 'Bad Credit']
    elif name == 'spambase':
        ds = fetch_ucirepo(id=94)
        X  = ds.data.features
        y  = ds.data.targets.iloc[:, 0]
        class_names = ['Not Spam', 'Spam']
    else:
        raise ValueError(f"Unknown dataset: {name}")

    X = pd.get_dummies(X, drop_first=True)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
    y = y.apply(pd.to_numeric, errors='coerce').fillna(0)

    X_tv, X_test, y_tv, y_test = train_test_split(
        X.values, y.values, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.15, random_state=42, stratify=y_tv)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    lower = torch.tensor(X_train.min(axis=0), dtype=torch.float32).to(DEVICE)
    upper = torch.tensor(X_train.max(axis=0), dtype=torch.float32).to(DEVICE)

    return (TabularDataset(X_train, y_train),
            TabularDataset(X_val,   y_val),
            TabularDataset(X_test,  y_test),
            scaler, X.columns.tolist(), class_names, lower, upper)


#Defining the models

class MCDropoutMLP(nn.Module):
    """MC Dropout model used as base model and for NERCE training."""
    def __init__(self, input_dim, num_classes, hidden_dim1=64,
                 hidden_dim2=32, dropout_rate=0.2):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim1)
        self.drop1 = nn.Dropout(p=dropout_rate)
        self.fc2   = nn.Linear(hidden_dim1, hidden_dim2)
        self.drop2 = nn.Dropout(p=dropout_rate)
        self.fc3   = nn.Linear(hidden_dim2, num_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x));  x = self.drop1(x)
        x = F.relu(self.fc2(x));  x = self.drop2(x)
        return F.log_softmax(self.fc3(x), dim=-1)


class StandardMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_classes), nn.LogSoftmax(dim=-1)
        )
    def forward(self, x): return self.net(x)


class AutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=10, hidden_dim=40):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))


class TabularCVAE(nn.Module):
    def __init__(self, input_dim, num_classes, latent_dim=10, hidden_dim=64):
        super().__init__()
        self.input_dim   = input_dim
        self.num_classes = num_classes
        self.latent_dim  = latent_dim
        self.label_emb   = nn.Embedding(num_classes, num_classes)
        self.enc_fc1  = nn.Linear(input_dim + num_classes, hidden_dim)
        self.enc_mu   = nn.Linear(hidden_dim, latent_dim)
        self.enc_logv = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1  = nn.Linear(latent_dim + num_classes, hidden_dim)
        self.dec_fc2  = nn.Linear(hidden_dim, input_dim)

    def encode(self, x, y):
        h = F.relu(self.enc_fc1(torch.cat([x, self.label_emb(y)], dim=1)))
        return self.enc_mu(h), self.enc_logv(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z, y):
        h = F.relu(self.dec_fc1(torch.cat([z, self.label_emb(y)], dim=1)))
        return self.dec_fc2(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar



# Rashomon Ensemble Approx (MC Dropout Frozen Masks)

class FrozenMaskMLP(nn.Module):
    """A deterministic sub-network sampled from the MC Dropout posterior."""
    def __init__(self, base_model, p=0.2):
        super().__init__()
        # Deep copy to safely freeze weights
        self.base = copy.deepcopy(base_model)
        self.base.eval() # Disable native PyTorch dropout

        # Pre-sample binary masks, scaled for inference parity
        self.mask1 = (torch.rand(self.base.fc1.out_features) > p).float() / (1.0 - p)
        self.mask2 = (torch.rand(self.base.fc2.out_features) > p).float() / (1.0 - p)

    def forward(self, x):
        x = F.relu(self.base.fc1(x))
        x = x * self.mask1.to(x.device)
        x = F.relu(self.base.fc2(x))
        x = x * self.mask2.to(x.device)
        return F.log_softmax(self.base.fc3(x), dim=-1)


class MCDropoutRashomonSet:
    """Samples N models from MC Dropout and filters them via the epsilon threshold."""
    def __init__(self, base_model, n_samples=50):
        self.models = [FrozenMaskMLP(base_model).to(DEVICE) for _ in range(n_samples)]
        self.val_losses = np.full(n_samples, np.inf)

    def evaluate_all(self, val_loader):
        print(f"  Evaluating {len(self.models)} MC Dropout masks on validation set...")
        ce = nn.NLLLoss()
        for i, model in enumerate(self.models):
            total, n = 0.0, 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y  = x.to(DEVICE), y.to(DEVICE)
                    total += ce(model(x), y).item() * len(y)
                    n     += len(y)
            self.val_losses[i] = total / n
        print(f"  Sampled val losses  min={self.val_losses.min():.4f}  "
              f"max={self.val_losses.max():.4f}  mean={self.val_losses.mean():.4f}")

    def get_rashomon_models(self, epsilon):
        L_star = self.val_losses.min()
        in_set = [m for m, l in zip(self.models, self.val_losses)
                  if l <= L_star + epsilon]
        # Fallback to the single best mask if epsilon is extremely tight
        return in_set if in_set else [self.models[int(np.argmin(self.val_losses))]]

    def rashomon_size(self, epsilon):
        return int((self.val_losses <= self.val_losses.min() + epsilon).sum())

    def expected_log_prob_rashomon(self, x_prime, target_cf, epsilon):
        """Approximates E_{theta~P_R}[log P_theta] using ONLY accepted masks."""
        members      = self.get_rashomon_models(epsilon)
        log_prob_sum = 0.0
        for m in members:
            # m is already a frozen, deterministic mask
            lp = m(x_prime).gather(1, target_cf.view(-1, 1)).squeeze()
            log_prob_sum = log_prob_sum + lp
        return (log_prob_sum / len(members)).mean()



# Helprs for model training

def train_mc_dropout(model, train_loader, epochs, name="MC Dropout"):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    ce  = nn.NLLLoss()
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); ce(model(x), y).backward(); opt.step()


def train_standard_mlp(model, train_loader, epochs, name="MLP"):
    opt = optim.Adam(model.parameters(), lr=0.001)
    ce  = nn.NLLLoss()
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); ce(model(x), y).backward(); opt.step()


def train_autoencoder(model, loader, epochs, name="AE"):
    opt = optim.Adam(model.parameters(), lr=0.001)
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad(); F.mse_loss(model(x), x).backward(); opt.step()

#My optimization fn.
def train_cvae_rashomon(cvae, train_loader, ensemble, epsilon, epochs=50,
                        lambda_class=1.0, lambda_prox=0.1, lambda_kl=0.1,
                        name="CVAE"):
    """Trains CVAE using the Rashomon-restricted ELBO-like objective."""
    opt = optim.Adam(cvae.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training {name} (eps={epsilon})...")
    for _ in range(epochs):
        cvae.train()
        for x, y in train_loader:
            x, y       = x.to(DEVICE), y.to(DEVICE)
            target_cf  = 1 - y
            mu, logvar = cvae.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z          = cvae.reparameterize(mu, logvar)
            x_prime    = cvae.decode(z, target_cf)
            exp_lp     = ensemble.expected_log_prob_rashomon(x_prime, target_cf, epsilon)
            proximity  = F.mse_loss(x_prime, x, reduction='mean')
            loss       = -(lambda_class * exp_lp) + (lambda_kl * kl_div) + (lambda_prox * proximity)
            opt.zero_grad(); loss.backward(); opt.step()


def train_cvae_standard(cvae, train_loader, epochs=50, name="CF-VAE"):
    """Plain CVAE — no Rashomon weighting."""
    opt = optim.Adam(cvae.parameters(), lr=0.001)
    print(f"  Training {name}...")
    for _ in range(epochs):
        cvae.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            recon, mu, logvar = cvae(x, y)
            mse = F.mse_loss(recon, x, reduction='sum')
            kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            (mse + kl).backward()
            opt.step()


#Thisis actually the original version before epsilon - not used.
def train_cvae_nerce(cvae, train_loader, base_model, epochs=50,
                     lambda_class=1.0, lambda_prox=0.1, lambda_kl=0.1,
                     name="MM-CF-CVAE"):
    """
    Trains the CVAE using MC Dropout to approximate E_{theta~P(theta|D)}.
    Original NERCE formulation — no epsilon restriction on the posterior.
    """
    opt = optim.Adam(cvae.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training {name}...")
    for _ in range(epochs):
        cvae.train()
        for x, y in train_loader:
            x, y       = x.to(DEVICE), y.to(DEVICE)
            target_cf  = 1 - y
            mu, logvar = cvae.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z          = cvae.reparameterize(mu, logvar)
            x_prime    = cvae.decode(z, target_cf)
            base_model.train()
            M      = 10
            exp_lp = sum(
                base_model(x_prime).gather(1, target_cf.view(-1, 1)).squeeze()
                for _ in range(M)
            ) / M
            exp_lp    = exp_lp.mean()
            proximity = F.mse_loss(x_prime, x, reduction='mean')
            loss      = -(lambda_class * exp_lp) + (lambda_kl * kl_div) + (lambda_prox * proximity)
            opt.zero_grad(); loss.backward(); opt.step()


# Generative methods for CF's

#gen cf amortized is for the CVAE-like methods, for target class y_t
def generate_cf_amortized(cvae, x_original, target_class, lower, upper, n_samples=5):
    """Single forward pass through trained CVAE; pick lowest-proximity sample."""
    cvae.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        y_t        = torch.tensor([target_class], device=DEVICE)
        mu, logvar = cvae.encode(x_original, y_t)
        candidates = [cvae.decode(cvae.reparameterize(mu, logvar), y_t).clamp(lower, upper)
                      for _ in range(n_samples)]
        dists   = [torch.linalg.norm(c - x_original).item() for c in candidates]
        x_prime = candidates[int(np.argmin(dists))]
    return x_prime, time.perf_counter() - t0


#this is an implementation of Schut et al. - note that they don't use MC dropout, so it's adapated to use avg class score.
def generate_cf_greedy(model, x_original, target_class, lower, upper):
    """Feature-wise greedy gradient baseline."""
    t0               = time.perf_counter()
    x_prime          = x_original.clone().detach()
    altered_features = torch.zeros_like(x_prime, device=DEVICE)
    for _ in range(2000):
        x_prime.requires_grad_(True)
        model.train()
        x_rep    = x_prime.repeat(100, 1)
        logits   = model(x_rep)
        tgt_rep  = torch.tensor([target_class] * 100, device=DEVICE)
        avg_loss = F.nll_loss(logits, tgt_rep, reduction='none').mean()
        avg_loss.backward()
        grad    = x_prime.grad.clone().detach()
        x_prime = x_prime.detach()
        with torch.no_grad():
            avg_probs = torch.exp(logits).mean(dim=0)
            conf, pred = torch.max(avg_probs, dim=0)
            if pred.item() == target_class and conf.item() >= 0.95:
                break
            grad_flat    = grad.view(-1)
            altered_flat = altered_features.view(-1)
            grad_flat[altered_flat >= 5] = 0.0
            if grad_flat.abs().max() == 0:
                break
            max_idx = torch.argmax(grad_flat.abs())
            x_prime.view(-1)[max_idx] -= 0.1 * grad_flat[max_idx].sign()
            altered_features.view(-1)[max_idx] += 1
            x_prime.clamp_(lower, upper)
    return x_prime.detach(), time.perf_counter() - t0


#Implementation of Batten et al.
def generate_cf_paper_bnn(model, x_original, target_class, lower, upper):
    """
    Per-instance gradient descent using MC Dropout.
    Matches original paper hyperparameters: 2001 steps, lr=0.1, n_mc=100.
    """
    t0      = time.perf_counter()
    x_prime = x_original.clone().detach().requires_grad_(True)
    opt     = optim.Adam([x_prime], lr=0.1)
    for _ in range(2001):
        opt.zero_grad()
        model.train()
        _, _, mean_lp = get_bayesian_prediction_stats(model, x_prime, target_class)
        loss = -mean_lp + 0.001 * torch.linalg.norm(x_prime - x_original)
        loss.backward()
        opt.step()
        with torch.no_grad():
            x_prime.clamp_(lower, upper)
        model.train()
        with torch.no_grad():
            outs = torch.stack([model(x_prime) for _ in range(20)]).mean(0)
            if outs.max(1)[1].item() == target_class:
                break
    return x_prime.detach(), time.perf_counter() - t0

def generate_cf_quce(model, ae, x_original, target_class, lower, upper):
    """Latent-space gradient optimisation through a general autoencoder - can replace with VAE."""
    t0 = time.perf_counter()
    ae.eval()
    with torch.no_grad():
        z0 = ae.encoder(x_original)
    z_prime = z0.clone().detach().requires_grad_(True)
    opt     = optim.Adam([z_prime], lr=0.1)
    for _ in range(2001):
        opt.zero_grad()
        x_prime   = ae.decoder(z_prime)
        _, _, mlp = get_bayesian_prediction_stats(model, x_prime, target_class)
        recon_l   = F.mse_loss(ae(x_prime), x_prime)
        prox_l    = F.mse_loss(x_prime, x_original)
        (0.6 * (-mlp) + 0.2 * recon_l + 0.2 * prox_l).backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            if model(x_prime).max(1)[1].item() == target_class:
                break
    final = ae.decoder(z_prime).detach().clamp(lower, upper)
    return final, time.perf_counter() - t0


#diff metrics that can be used..
def validity(cf, target_class, model, n_mc=20):
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(cf) for _ in range(n_mc)]).mean(0)
    return int(outs.max(1)[1].item() == target_class)


def l2_distance(cf, x_orig):
    return torch.linalg.norm(cf - x_orig).item()


def im1_score(cf, target_class, original_class, ae_by_class):
    ae_t = ae_by_class.get(target_class)
    ae_o = ae_by_class.get(original_class)
    if not ae_t or not ae_o:
        return np.inf
    ae_t.eval(); ae_o.eval()
    with torch.no_grad():
        err_t = torch.linalg.norm(cf - ae_t(cf)).item() ** 2
        err_o = torch.linalg.norm(cf - ae_o(cf)).item() ** 2
    return err_t / (err_o + 1e-8)


def lof_score(cf, lof_model):
    return -lof_model.score_samples(
        cf.detach().cpu().numpy().reshape(1, -1)).mean()


def implausibility(cf, in_class_samples):
    cf_cpu = cf.detach().cpu()
    if cf_cpu.dim() == 1:
        cf_cpu = cf_cpu.unsqueeze(0)
    return torch.cdist(cf_cpu, in_class_samples.cpu()).mean().item()

#This is surrogate model validity - NOT cross model validity
def cross_model_validity(cf, target_class, surrogate):
    surrogate.eval()
    with torch.no_grad():
        return int(surrogate(cf).max(1)[1].item() == target_class)

#This is actually cross model validity below, not validity after training
def validity_after_training(cf, target_class, base_model, n_masks=10):
    """
    Simulates model shift by drawing n_masks fresh FrozenMaskMLPs from the
    base MC Dropout model — independent of the Rashomon set used for training.
    Returns the fraction of fresh masks that predict y' for this CF.
    A CF that only fools its training-time model distribution but not new
    draws has low VAT, indicating fragility to model retraining.
    """
    votes = 0
    for _ in range(n_masks):
        fresh_mask = FrozenMaskMLP(base_model).to(cf.device)
        with torch.no_grad():
            votes += int(fresh_mask(cf).max(1)[1].item() == target_class)
    return votes / n_masks


def robustness_ne(cf, target_class, model, noise=0.01, n_mc=20, n_noise=20):
    model.train()
    with torch.no_grad():
        base = torch.exp(torch.stack([model(cf) for _ in range(n_mc)]).mean(0))
    diffs = []
    for _ in range(n_noise):
        noisy = cf + torch.randn_like(cf) * noise
        with torch.no_grad():
            pert = torch.exp(torch.stack([model(noisy) for _ in range(n_mc)]).mean(0))
        diffs.append(torch.linalg.norm(pert - base).item() ** 2)
    return float(np.mean(diffs))



def robustness_ic(x_original, cf, generator_callable, noise_level=0.01):
    """
    Input Change Robustness (R_IC):
    Perturbs x_original slightly and re-generates a CF; measures how much
    the CF shifts relative to the original CF distance.
    R_IC = ||cf_perturbed - cf||^2 / (||cf - x_original||^2 + eps)
    Lower is better: a robust method produces similar CFs for similar inputs.
    generator_callable: lambda x -> (cf_tensor, time) with all kwargs captured.
    """
    x_pert = x_original + (torch.rand_like(x_original) - 0.5) * noise_level
    cf_pert, _ = generator_callable(x_pert)
    if cf_pert is None or not torch.all(torch.isfinite(cf_pert)):
        return np.nan
    l2_cf   = torch.linalg.norm(cf_pert - cf).item() ** 2
    l2_orig = torch.linalg.norm(cf - x_original).item() ** 2
    return l2_cf / (l2_orig + 1e-6)

def noisy_execution_variance(cf, target_class, model, noise=0.01, n_mc=50, n_noise=20):
    model.train()
    probs = []
    for _ in range(n_noise):
        noisy = cf + torch.randn_like(cf) * noise
        with torch.no_grad():
            p = torch.exp(model(noisy.repeat(n_mc, 1)))[:, target_class].mean().item()
        probs.append(p)
    return float(np.var(probs))


def rashomon_validity_ratio(cf, target_class, ensemble, epsilon):
    """Fraction of Rashomon-set members that predict y' for this CF."""
    members = ensemble.get_rashomon_models(epsilon)
    votes = 0
    for m in members:
        m.eval()
        with torch.no_grad():
            votes += int(m(cf).max(1)[1].item() == target_class)
    return votes / len(members)


def _collect_metrics(cf, target_class, pred_label, x_original,
                     base_model, surrogate,
                     ae_by_class, in_class_samples,
                     exec_time, ensemble=None, epsilon=None,
                     generator_callable=None):
    """Compute all metrics for one CF; returns a flat dict."""
    m = {}
    m["Validity"]               = validity(cf, target_class, base_model)
    m["IM1"]                    = im1_score(cf, target_class, pred_label, ae_by_class)
    m["Implausibility"]         = implausibility(cf, in_class_samples[target_class])
    m["Cross-Model Validity"]   = cross_model_validity(cf, target_class, surrogate)
    m["Validity After Training"]= validity_after_training(cf, target_class, base_model)
    for _nl in [0.1, 0.01, 0.001]:
        m[f"NE Robustness (s={_nl})"] = robustness_ne(cf, target_class, base_model, noise=_nl)
    m["Inference Time (s)"]     = exec_time
    if generator_callable is not None:
        for _nl in [0.1, 0.01, 0.001]:
            m[f"R_IC (s={_nl})"] = robustness_ic(x_original, cf, generator_callable, noise_level=_nl)
    else:
        for _nl in [0.1, 0.01, 0.001]:
            m[f"R_IC (s={_nl})"] = np.nan
    if ensemble is not None and epsilon is not None:
        m["Rashomon Validity Ratio"] = rashomon_validity_ratio(
            cf, target_class, ensemble, epsilon)
    else:
        m["Rashomon Validity Ratio"] = np.nan
    return m


# run across deterministic epsilons
def run_epsilon_experiment(dataset_name='credit',
                           epsilons=(0.05, 0.1, 0.2, 0.4, 0.8),
                           seeds=(11, 22, 33),
                           n_ensemble=20,
                           n_samples_eval=60,
                           cvae_epochs=50,
                           ensemble_epochs=40):
    """
    For each epsilon:
      - Trains a Rashomon CVAE (epsilon-specific)
    Once per seed (epsilon-agnostic):
      - CF-VAE, MM-CF-CVAE, Batten et al., QUCE, Schut et al.
    All methods evaluated on the same test instances; baselines fanned
    into every epsilon row so comparisons are on identical samples.
    """
    results = {eps: defaultdict(lambda: defaultdict(list))
               for eps in epsilons}

    for seed in seeds:
        set_seed(seed)
        print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")

        # ---- Data ----
        (train_ds, val_ds, test_ds,
         scaler, feat_names, class_names,
         lower, upper) = get_dataset(dataset_name)

        INPUT_DIM   = train_ds.features.shape[1]
        NUM_CLASSES = len(class_names)
        LATENT_DIM  = max(5, INPUT_DIM // 4)

        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
        test_loader  = DataLoader(test_ds,  batch_size=1,   shuffle=False)

        # ---- Shared models ----
        print("\n[1/7] MC Dropout base model (Unrestricted MM-CF-CVAE backbone)...")
        base_model = MCDropoutMLP(INPUT_DIM, NUM_CLASSES).to(DEVICE)
        train_mc_dropout(base_model, train_loader, epochs=50)

        print("\n[2/7] Generating Restricted Rashomon Set from MC Dropout...")
        ensemble = MCDropoutRashomonSet(base_model, n_samples=n_ensemble)
        ensemble.evaluate_all(val_loader)

        print("\n[3/7] Surrogate MLPs (CMV)...")
        surrogate = StandardMLP(INPUT_DIM, NUM_CLASSES).to(DEVICE)
        train_standard_mlp(surrogate, train_loader, epochs=30, name="CMV surrogate")

        print("\n[4/7] Autoencoders (IM1 + QUCE)...")
        ae_general = AutoEncoder(INPUT_DIM, LATENT_DIM).to(DEVICE)
        train_autoencoder(ae_general, train_loader, epochs=40, name="General AE")
        ae_by_class = {}
        for c in range(NUM_CLASSES):
            idxs     = [i for i, lbl in enumerate(train_ds.labels) if lbl == c]
            c_loader = DataLoader(Subset(train_ds, idxs), batch_size=32, shuffle=True)
            ae_c     = AutoEncoder(INPUT_DIM, LATENT_DIM).to(DEVICE)
            train_autoencoder(ae_c, c_loader, epochs=40, name=f"AE class {c}")
            ae_by_class[c] = ae_c

        train_feats  = train_ds.features.cpu()
        train_labels = train_ds.labels.cpu()
        in_class_samp = {c: train_feats[train_labels == c] for c in range(NUM_CLASSES)}

        print("\n[5/7] CF-VAE baseline...")
        cvae_std = TabularCVAE(INPUT_DIM, NUM_CLASSES, LATENT_DIM).to(DEVICE)
        train_cvae_standard(cvae_std, train_loader, epochs=cvae_epochs)

        print("\n[6/7] MM-CF-CVAE CVAE (MC Dropout, no epsilon restriction)...")
        cvae_nerce = TabularCVAE(INPUT_DIM, NUM_CLASSES, LATENT_DIM).to(DEVICE)
        train_cvae_nerce(cvae_nerce, train_loader, base_model, epochs=cvae_epochs)

        print("\n[7/7] Rashomon CVAEs — one per epsilon...")
        cvae_by_eps = {}
        for eps in epsilons:
            cvae_r = TabularCVAE(INPUT_DIM, NUM_CLASSES, LATENT_DIM).to(DEVICE)
            train_cvae_rashomon(cvae_r, train_loader, ensemble, eps,
                                epochs=cvae_epochs,
                                name=f"Rashomon CVAE (eps={eps})")
            cvae_by_eps[eps] = cvae_r

        # ---- Evaluation loop ----
        print("\n--- Evaluating ---")
        base_model.train()
        processed = 0

        for x, y in test_loader:
            if processed >= n_samples_eval:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.no_grad():
                outs       = torch.stack([base_model(x) for _ in range(20)]).mean(0)
                pred_label = outs.max(1)[1].item()
            if pred_label != y.item():
                continue
            processed += 1
            target_cf = 1 - pred_label

            # Epsilon-agnostic baselines: generate once - run models that don't require epsilon
            cf_std,   t_std   = generate_cf_amortized(cvae_std,   x, target_cf, lower, upper)
            cf_nerce, t_nerce = generate_cf_amortized(cvae_nerce, x, target_cf, lower, upper)
            cf_bnn,   t_bnn   = generate_cf_paper_bnn(base_model,  x, target_cf, lower, upper)
            cf_quce,  t_quce  = generate_cf_quce(base_model, ae_general, x, target_cf, lower, upper)
            cf_grdy,  t_grdy  = generate_cf_greedy(base_model,    x, target_cf, lower, upper)

            # Generator callables for R_IC — capture all kwargs so only x varies (robustness to input change metric)
            gen_std   = lambda xp: generate_cf_amortized(cvae_std,   xp, target_cf, lower, upper)
            gen_nerce = lambda xp: generate_cf_amortized(cvae_nerce, xp, target_cf, lower, upper)
            gen_bnn   = lambda xp: generate_cf_paper_bnn(base_model,  xp, target_cf, lower, upper)
            gen_quce  = lambda xp: generate_cf_quce(base_model, ae_general, xp, target_cf, lower, upper)
            gen_grdy  = lambda xp: generate_cf_greedy(base_model,    xp, target_cf, lower, upper)

            baseline_cfs = [
                ("CF-VAE",        cf_std,   t_std,   gen_std),
                ("MM-CF-CVAE",    cf_nerce, t_nerce, gen_nerce),
                ("Batten et al.", cf_bnn,   t_bnn,   gen_bnn),
                ("QUCE",          cf_quce,  t_quce,  gen_quce),
                ("Schut et al.",  cf_grdy,  t_grdy,  gen_grdy),
            ]

            # Fan baseline results into every epsilon row
            for eps in epsilons:
                for mname, cf, et, gen_fn in baseline_cfs:
                    if cf is None or not torch.all(torch.isfinite(cf)):
                        continue
                    m = _collect_metrics(
                        cf, target_cf, pred_label, x,
                        base_model, surrogate,
                        ae_by_class, in_class_samp,
                        et, ensemble=ensemble, epsilon=eps,
                        generator_callable=gen_fn)
                    for k, v in m.items():
                        results[eps][mname][k].append(v)

                #Rashomon CVAE for this epsilon
                _cvae_r  = cvae_by_eps[eps]
                gen_rash = lambda xp, _c=_cvae_r: generate_cf_amortized(_c, xp, target_cf, lower, upper)
                cf_r, t_r = generate_cf_amortized(_cvae_r, x, target_cf, lower, upper)
                if cf_r is not None and torch.all(torch.isfinite(cf_r)):
                    rname = f"Rashomon CVAE (eps={eps})"
                    m = _collect_metrics(
                        cf_r, target_cf, pred_label, x,
                        base_model, surrogate,
                        ae_by_class, in_class_samp,
                        t_r, ensemble=ensemble, epsilon=eps,
                        generator_callable=gen_rash)
                    for k, v in m.items():
                        results[eps][rname][k].append(v)

        print(f"    Evaluated on {processed} instances.")

    # Aggregate the results across seeds - for mean and std of metrics across epsilon
    summary_rows = []
    for eps in epsilons:
        for method, metric_dict in results[eps].items():
            row = {"Epsilon": eps, "Method": method}
            for metric, vals in metric_dict.items():
                finite = [v for v in vals if np.isfinite(v)]
                row[metric]          = np.mean(finite) if finite else np.nan
                row[f"{metric}_std"] = np.std(finite)  if finite else np.nan
            summary_rows.append(row)

    return pd.DataFrame(summary_rows)


# Plots

METRICS_TO_PLOT = [
    ("Validity",                 "higher is better", True),
    ("IM1",                      "lower is better",  False),
    ("Implausibility",           "lower is better",  False),
    ("Cross-Model Validity",     "higher is better", True),
    ("Validity After Training",  "higher is better", True),
    ("NE Robustness (s=0.1)",    "lower is better",  False),
    ("NE Robustness (s=0.01)",   "lower is better",  False),
    ("NE Robustness (s=0.001)",  "lower is better",  False),
    ("R_IC (s=0.1)",             "lower is better",  False),
    ("R_IC (s=0.01)",            "lower is better",  False),
    ("R_IC (s=0.001)",           "lower is better",  False),
    ("Rashomon Validity Ratio",  "higher is better", True),
    ("Inference Time (s)",       "lower is better",  False),
]

BASELINE_STYLES = {
    "CF-VAE":   dict(color="#5e81ac", linestyle="--"),
    "MM-CF-CVAE": dict(color="#a3be8c", linestyle="--"),
    "Batten et al.":     dict(color="#ebcb8b", linestyle=":"),
    "QUCE":            dict(color="#d08770", linestyle=":"),
    "Schut et al.":          dict(color="#b48ead", linestyle="-."),
}


def plot_epsilon_sweep(df, dataset_name, out_dir="."):
    os.makedirs(out_dir, exist_ok=True)

    epsilons    = sorted(df["Epsilon"].unique())
    all_methods = df["Method"].unique().tolist()
    rash_meths  = sorted([m for m in all_methods if "Rashomon CVAE" in m],
                         key=lambda m: float(m.split("eps=")[1].rstrip(")")))
    base_meths  = [m for m in all_methods if "Rashomon CVAE" not in m]

    ncols = 3
    nrows = (len(METRICS_TO_PLOT) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(6 * ncols, 4.5 * nrows),
                             facecolor="#1a1a2e")
    fig.suptitle(f"Rashomon eps Sweep  -  {dataset_name.replace('_', ' ').title()}",
                 fontsize=18, fontweight="bold", color="white", y=1.01)

    for ax in axes.flat:
        ax.set_facecolor("#16213e")
        ax.tick_params(colors="#e0e0e0")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

    cmap      = plt.cm.viridis
    eps_color = {e: cmap(i / max(len(epsilons) - 1, 1))
                 for i, e in enumerate(sorted(epsilons))}

    for idx, (metric, label, _) in enumerate(METRICS_TO_PLOT):
        ax = axes.flat[idx]
        ax.set_title(metric, color="white", fontsize=11, pad=6)
        ax.set_xlabel("eps",  color="#a0a0c0", fontsize=10)
        ax.set_ylabel(label,  color="#a0a0c0", fontsize=9)

        # Rashomon CVAE line
        rash_pts = []
        for meth in rash_meths:
            eps_val = float(meth.split("eps=")[1].rstrip(")"))
            sub = df[df["Method"] == meth]
            if sub.empty or metric not in sub.columns:
                continue
            yval = sub[metric].values[0]
            ystd = sub[f"{metric}_std"].values[0] if f"{metric}_std" in sub.columns else 0
            rash_pts.append((eps_val, yval, ystd))
            ax.errorbar([eps_val], [yval],
                        yerr=[[ystd]] if np.isfinite(ystd) and ystd > 0 else None,
                        fmt="o", color=eps_color[eps_val],
                        ecolor=eps_color[eps_val],
                        capsize=3, markersize=6,
                        label="_nolegend_")
        if rash_pts:
            xs, ys, _ = zip(*sorted(rash_pts))
            ax.plot(xs, ys, color="#a8d8ea", linewidth=1.5, label="_nolegend_")

        # Baselines as horizontal lines
        for bm in base_meths:
            # Use epsilon=epsilons[0] row (all eps rows are identical for baselines)
            sub = df[(df["Method"] == bm) & (df["Epsilon"] == epsilons[0])]
            if sub.empty or metric not in sub.columns:
                continue
            bval = sub[metric].values[0]
            if not np.isfinite(bval):
                continue
            style = BASELINE_STYLES.get(bm, dict(color="grey", linestyle="--"))
            ax.axhline(bval, color=style["color"], linestyle=style["linestyle"],
                       linewidth=1.5, alpha=0.85,
                       label=bm if idx == 0 else "_nolegend_")

        ax.grid(True, color="#2a2a4a", linestyle=":", linewidth=0.7)

    for ax in axes.flat[len(METRICS_TO_PLOT):]:
        ax.set_visible(False)

    # Legend
    handles, labels = [], []
    handles.append(Line2D([0], [0], color="#a8d8ea", linewidth=2,
                          marker="o", markerfacecolor="#a8d8ea"))
    labels.append("Rashomon CVAE (sweep)")
    for bm in base_meths:
        style = BASELINE_STYLES.get(bm, dict(color="grey", linestyle="--"))
        handles.append(Line2D([0], [0], color=style["color"],
                               linestyle=style["linestyle"], linewidth=2))
        labels.append(bm)

    fig.legend(handles, labels,
               loc="lower center", ncol=3,
               frameon=True, framealpha=0.15,
               facecolor="#1a1a2e", edgecolor="#444",
               labelcolor="white", fontsize=9,
               bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout()
    path = os.path.join(out_dir, f"epsilon_sweep_{dataset_name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"\nFigure saved to {path}")
    plt.close(fig)


def plot_rashomon_size(ensemble, epsilons, dataset_name, out_dir="."):
    sizes = [ensemble.rashomon_size(e) for e in epsilons]
    fig, ax = plt.subplots(figsize=(7, 3.5), facecolor="#1a1a2e")
    ax.set_facecolor("#16213e")
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(epsilons)))
    bars   = ax.bar(range(len(epsilons)), sizes, color=colors,
                    edgecolor="#a8d8ea", linewidth=0.8)
    ax.set_xticks(range(len(epsilons)))
    ax.set_xticklabels([f"eps={e}" for e in epsilons], color="white")
    ax.set_ylabel("# Masks in Rashomon Set", color="#a0a0c0")
    ax.set_title(f"Rashomon Set Size vs eps  -  {dataset_name}",
                 color="white", fontsize=12)
    ax.tick_params(colors="#e0e0e0")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    ax.grid(axis='y', color="#2a2a4a", linestyle=":", linewidth=0.7)
    for bar, sz in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                str(sz), ha='center', va='bottom', color='white', fontsize=9)
    plt.tight_layout()
    path = os.path.join(out_dir, f"rashomon_size_{dataset_name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"Figure saved to {path}")
    plt.close(fig)


def save_results_table(df, dataset_name, out_dir="."):
    metric_cols = [m for m, _, _ in METRICS_TO_PLOT if m in df.columns]
    mean_cols   = ["Epsilon", "Method"] + metric_cols
    clean       = df[mean_cols].copy()
    for col in metric_cols:
        clean[col] = clean[col].apply(lambda v: f"{v:.4f}" if pd.notna(v) else "N/A")

    csv_path = os.path.join(out_dir, f"results_{dataset_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nFull results CSV saved to {csv_path}")

    txt_path = os.path.join(out_dir, f"results_{dataset_name}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"RASHOMON EPSILON SWEEP RESULTS - {dataset_name.upper()}\n")
        f.write("=" * 80 + "\n\n")
        for eps in sorted(df["Epsilon"].unique()):
            f.write(f"  eps = {eps}\n{'-'*60}\n")
            sub = clean[clean["Epsilon"] == eps].drop(columns="Epsilon")
            f.write(sub.to_string(index=False))
            f.write("\n\n")
    print(f"Readable table saved to {txt_path}")


#Experiments - Main - one can change the datasets from UCI repo (I hope)

if __name__ == "__main__":
    DATASET      = "adult_income"       # 'adult_income' | 'credit' | 'spambase'
    EPSILONS     = [0.01, 0.05, 0.1, 0.2, 0.4, 0.8]
    SEEDS        = [11, 22, 33, 44, 55]
    N_ENSEMBLE   = 50
    N_EVAL       = 100
    CVAE_EPOCHS  = 50
    ENS_EPOCHS   = 50
    OUT_DIR      = "./rashomon_results"

    os.makedirs(OUT_DIR, exist_ok=True)

    results_df = run_epsilon_experiment(
        dataset_name    = DATASET,
        epsilons        = EPSILONS,
        seeds           = SEEDS,
        n_ensemble      = N_ENSEMBLE,
        n_samples_eval  = N_EVAL,
        cvae_epochs     = CVAE_EPOCHS,
        ensemble_epochs = ENS_EPOCHS,
    )

    save_results_table(results_df, DATASET, OUT_DIR)s
    plot_epsilon_sweep(results_df, DATASET, OUT_DIR)

    # Rashomon size plot (retrain base model on last seed for the figure)
    set_seed(SEEDS[-1])
    train_ds, val_ds, *_ = get_dataset(DATASET)
    tl  = DataLoader(train_ds, batch_size=64,  shuffle=True)
    vl  = DataLoader(val_ds,   batch_size=256, shuffle=False)

    # Train the base model, then extract the masks
    base = MCDropoutMLP(train_ds.features.shape[1], 2).to(DEVICE)
    train_mc_dropout(base, tl, epochs=ENS_EPOCHS)

    ens = MCDropoutRashomonSet(base, n_samples=N_ENSEMBLE)
    ens.evaluate_all(vl)

    plot_rashomon_size(ens, EPSILONS, DATASET, OUT_DIR)

    print("\nAll done. Results in", OUT_DIR)

In [ ]:
"""
Rashomon Epsilon Sweep Experiment (MC Dropout Version) - PneumoniaMNIST CNN
===========================================================================
Integrates the Rashomon-restricted posterior P_R(theta) sweep with a
spatial CNN backbone and MedMNIST dataloaders.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, SubsetRandomSampler
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
import random
import time
import copy
from collections import defaultdict

import medmnist
from medmnist import INFO
from sklearn.neighbors import LocalOutlierFactor


# setup on cuda
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_bayesian_prediction_stats(model, x, target_class, n_samples=100):
    model.train()
    x = x.to(DEVICE)
    batch_size = x.shape[0]
    x_rep = x.repeat(n_samples, 1, 1, 1)
    preds = model(x_rep).view(n_samples, batch_size, -1)
    log_probs = preds[:, :, target_class]
    mean_log_prob = log_probs.mean()
    mean_prob = torch.exp(log_probs).mean()
    variance = torch.exp(log_probs).var()
    return mean_prob, variance, mean_log_prob


# Data import / handling

def get_pneumoniamnist_data():
    print("\n--- Loading PneumoniaMNIST dataset ---")
    data_flag = 'pneumoniamnist'
    info = INFO[data_flag]
    DataClass = getattr(medmnist, info['python_class'])

    transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = DataClass(split='train', transform=transform, download=True)
    val_dataset   = DataClass(split='val',   transform=transform, download=True)
    test_dataset  = DataClass(split='test',  transform=transform, download=True)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=1,   shuffle=False)

    class_names = ['Normal', 'Pneumonia']
    lower_bound, upper_bound = 0.0, 1.0

    return train_loader, val_loader, test_loader, class_names, lower_bound, upper_bound


#Model Definitions

class MCDropoutCNN(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.drop1 = nn.Dropout2d(p=dropout_rate)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.drop2 = nn.Dropout2d(p=dropout_rate)
        self.fc1   = nn.Linear(320, 50)
        self.drop3 = nn.Dropout(p=dropout_rate)
        self.fc2   = nn.Linear(50, num_classes)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = self.drop1(x)
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = self.drop2(x)
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = self.drop3(x)
        return F.log_softmax(self.fc2(x), dim=-1)

class StandardCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.fc1   = nn.Linear(320, 50)
        self.fc2   = nn.Linear(50, num_classes)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=-1)

class ImageAutoEncoder(nn.Module):
    def __init__(self, input_dim=784, latent_dim=20, hidden_dim=200):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid()
        )
    def forward(self, x):
        x_flat = x.view(x.size(0), -1)
        return self.decoder(self.encoder(x_flat)).view(x.size(0), 1, 28, 28)

class ImageCVAE(nn.Module):
    def __init__(self, num_classes=2, latent_dim=20, hidden_dim=400):
        super().__init__()
        self.num_classes = num_classes
        self.label_embedding = nn.Embedding(num_classes, 50)
        self.enc_fc1  = nn.Linear(784 + 50, hidden_dim)
        self.enc_fc21 = nn.Linear(hidden_dim, latent_dim)
        self.enc_fc22 = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1  = nn.Linear(latent_dim + 50, hidden_dim)
        self.dec_fc2  = nn.Linear(hidden_dim, 784)

    def encode(self, x, y):
        y_emb = self.label_embedding(y)
        inputs = torch.cat([x.view(x.size(0), -1), y_emb], dim=1)
        h1 = F.relu(self.enc_fc1(inputs))
        return self.enc_fc21(h1), self.enc_fc22(h1)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z, y):
        inputs = torch.cat([z, self.label_embedding(y)], dim=1)
        h3 = F.relu(self.dec_fc1(inputs))
        return torch.sigmoid(self.dec_fc2(h3)).view(-1, 1, 28, 28)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar


# Rashomon Ensemble - Rashomon set is chedcked on the validation set.
class FrozenMaskCNN(nn.Module):
    def __init__(self, base_model, p=0.2):
        super().__init__()
        self.base = copy.deepcopy(base_model)
        self.base.eval()
        self.mask1 = (torch.rand(1, 10, 1, 1) > p).float() / (1.0 - p)
        self.mask2 = (torch.rand(1, 20, 1, 1) > p).float() / (1.0 - p)
        self.mask3 = (torch.rand(1, 50) > p).float() / (1.0 - p)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.base.conv1(x), 2))
        x = x * self.mask1.to(x.device)
        x = F.relu(F.max_pool2d(self.base.conv2(x), 2))
        x = x * self.mask2.to(x.device)
        x = x.view(-1, 320)
        x = F.relu(self.base.fc1(x))
        x = x * self.mask3.to(x.device)
        return F.log_softmax(self.base.fc2(x), dim=-1)

class MCDropoutRashomonSet:
    def __init__(self, base_model, n_samples=50):
        self.models = [FrozenMaskCNN(base_model).to(DEVICE) for _ in range(n_samples)]
        self.val_losses = np.full(n_samples, np.inf)

    def evaluate_all(self, val_loader):
        print(f"  Evaluating {len(self.models)} MC Dropout masks on validation set...")
        ce = nn.NLLLoss()
        for i, model in enumerate(self.models):
            total, n = 0.0, 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
                    total += ce(model(x), y).item() * len(y)
                    n += len(y)
            self.val_losses[i] = total / n

    def get_rashomon_models(self, epsilon):
        L_star = self.val_losses.min()
        in_set = [m for m, l in zip(self.models, self.val_losses) if l <= L_star + epsilon]
        return in_set if in_set else [self.models[int(np.argmin(self.val_losses))]]

    def expected_log_prob_rashomon(self, x_prime, target_cf, epsilon):
        members = self.get_rashomon_models(epsilon)
        log_prob_sum = 0.0
        for m in members:
            lp = m(x_prime).gather(1, target_cf.view(-1, 1)).squeeze()
            log_prob_sum = log_prob_sum + lp
        return (log_prob_sum / len(members)).mean()


# Helpers for training the diff models
def train_standard(model, loader, epochs, name):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    ce = nn.NLLLoss()
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            opt.zero_grad(); ce(model(x), y).backward(); opt.step()

def train_ae(model, loader, epochs, name):
    opt = optim.Adam(model.parameters(), lr=0.001)
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad(); F.mse_loss(model(x), x).backward(); opt.step()

def train_cvae_std(model, loader, epochs, name="CF-VAE"):
    opt = optim.Adam(model.parameters(), lr=0.001)
    print(f"  Training {name}...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            opt.zero_grad()
            recon, mu, logvar = model(x, y)
            bce = F.binary_cross_entropy(recon.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            (bce + kl).backward()
            opt.step()

#This is not used, it waseffectively the method without epsilon
def train_cvae_nerce(model, loader, base_model, epochs, lambda_class=15.0, lambda_prox=0.01, lambda_kl=0.1, name="MM-CF-CVAE"):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training {name}...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            target_cf = 1 - y
            mu, logvar = model.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z = model.reparameterize(mu, logvar)
            x_prime = model.decode(z, target_cf)

            base_model.train()
            M = 10
            exp_lp = sum(base_model(x_prime).gather(1, target_cf.view(-1, 1)).squeeze() for _ in range(M)) / M
            exp_lp = exp_lp.mean()

            prox = F.mse_loss(x_prime.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            loss = -(lambda_class * exp_lp) + (lambda_kl * kl_div) + (lambda_prox * prox)
            opt.zero_grad(); loss.backward(); opt.step()

def train_cvae_rashomon(model, loader, ensemble, epsilon, epochs, lambda_class=1.0, lambda_prox=0.05, lambda_kl=0.01):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training Rashomon CVAE (eps={epsilon})...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            target_cf = 1 - y
            mu, logvar = model.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z = model.reparameterize(mu, logvar)
            x_prime = model.decode(z, target_cf)

            exp_lp = ensemble.expected_log_prob_rashomon(x_prime, target_cf, epsilon)
            prox = F.mse_loss(x_prime.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            loss = -(lambda_class * exp_lp) + (lambda_kl * kl_div) + (lambda_prox * prox)
            opt.zero_grad(); loss.backward(); opt.step()

#COUNTERFACTUAL METHODS
#gen amortized is mine
def gen_amortized(cvae, x_orig, target_class):
    cvae.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        yt = torch.tensor([target_class], device=DEVICE)
        mu, _ = cvae.encode(x_orig, yt)
        x_prime = cvae.decode(mu, yt)
    return x_prime.detach(), time.perf_counter() - t0

def gen_paper_bnn(model, x_orig, target_class):
    t0 = time.perf_counter()
    x_prime = x_orig.clone().detach().requires_grad_(True)
    opt = optim.Adam([x_prime], lr=0.1)
    for _ in range(2001):
        opt.zero_grad()
        model.train()
        _, _, mlp = get_bayesian_prediction_stats(model, x_prime, target_class)
        prox = torch.linalg.norm(x_prime.view(-1) - x_orig.view(-1))
        loss = -mlp + 0.01 * prox
        loss.backward()
        opt.step()
        with torch.no_grad(): x_prime.clamp_(0.0, 1.0)
        model.train()
        with torch.no_grad():
            if torch.stack([model(x_prime) for _ in range(20)]).mean(0).max(1)[1].item() == target_class: break
    return x_prime.detach(), time.perf_counter() - t0

def gen_greedy(model, x_orig, target_class):
    t0 = time.perf_counter()
    x_prime = x_orig.clone().detach()
    altered = torch.zeros_like(x_prime, device=DEVICE)
    for _ in range(2000):
        x_prime.requires_grad_(True)
        model.train()
        logits = model(x_prime.repeat(100, 1, 1, 1))
        loss = F.nll_loss(logits, torch.tensor([target_class]*100, device=DEVICE), reduction='none').mean()
        loss.backward()
        grad = x_prime.grad.clone().detach()
        x_prime = x_prime.detach()

        with torch.no_grad():
            avg_p = torch.exp(logits).mean(0)
            if avg_p.max(0)[1].item() == target_class and avg_p.max(0)[0].item() >= 0.95: break

            gf = grad.view(-1)
            af = altered.view(-1)
            gf[af >= 20] = 0.0
            if gf.abs().max() == 0: break

            midx = torch.argmax(gf.abs())
            xp_flat = x_prime.view(-1)
            xp_flat[midx] -= 0.2 * gf[midx].sign()
            x_prime = xp_flat.view(1, 1, 28, 28).clamp(0.0, 1.0)
            af[midx] += 1
    return x_prime.detach(), time.perf_counter() - t0

def gen_quce(model, ae, x_orig, target_class):
    t0 = time.perf_counter()
    ae.eval()
    with torch.no_grad(): z0 = ae.encoder(x_orig.view(1, -1))
    z_prime = z0.clone().detach().requires_grad_(True)
    opt = optim.Adam([z_prime], lr=0.1)
    for _ in range(2001):
        opt.zero_grad()
        x_prime = ae.decoder(z_prime).view(1, 1, 28, 28).clamp(0.0, 1.0)
        _, _, mlp = get_bayesian_prediction_stats(model, x_prime, target_class)
        recon_l = F.mse_loss(ae(x_prime), x_prime)
        prox_l = F.mse_loss(x_prime, x_orig)
        (5.0 * (-mlp) + 0.5 * recon_l + 0.5 * prox_l).backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            if model(x_prime).max(1)[1].item() == target_class: break
    return ae.decoder(z_prime).detach().view(1, 1, 28, 28).clamp(0.0, 1.0), time.perf_counter() - t0


#evals
def _collect_metrics(cf, target, pred_label, orig, base_model, cmv, ae_class, in_class, et, ens=None, eps=None, gen_fn=None):
    m = {}
    base_model.train()
    with torch.no_grad(): m["Validity"] = int(torch.stack([base_model(cf) for _ in range(20)]).mean(0).max(1)[1] == target)

    cmv.eval()
    with torch.no_grad(): m["Cross-Model Validity"] = int(cmv(cf).max(1)[1] == target)

    votes = 0
    for _ in range(10):
        with torch.no_grad(): votes += int(FrozenMaskCNN(base_model).to(DEVICE)(cf).max(1)[1] == target)
    m["Validity After Training"] = votes / 10

    ae_t, ae_o = ae_class.get(target), ae_class.get(pred_label)
    if ae_t and ae_o:
        ae_t.eval(); ae_o.eval()
        with torch.no_grad():
            et_err = torch.linalg.norm(cf.view(-1) - ae_t(cf).view(-1)).item()**2
            eo_err = torch.linalg.norm(cf.view(-1) - ae_o(cf).view(-1)).item()**2
        m["IM1"] = et_err / (eo_err + 1e-8)
    else: m["IM1"] = np.nan

    if len(in_class[target]) > 0:
        m["Implausibility"] = torch.cdist(cf.view(1, -1).cpu(), in_class[target].cpu()).mean().item()
    else: m["Implausibility"] = np.nan

    m["Inference Time (s)"] = et

    for nl in [0.1, 0.01, 0.001]:
        base_model.train()
        diffs = []
        with torch.no_grad(): b_out = torch.exp(base_model(cf.repeat(20, 1, 1, 1))).mean(0)
        for _ in range(20):
            noisy = torch.clamp(cf + torch.randn_like(cf) * nl, 0.0, 1.0)
            with torch.no_grad(): n_out = torch.exp(base_model(noisy.repeat(20, 1, 1, 1))).mean(0)
            diffs.append(torch.linalg.norm(n_out - b_out).item()**2)
        m[f"NE Robustness (s={nl})"] = np.mean(diffs)

        if gen_fn:
            x_pert = torch.clamp(orig + (torch.rand_like(orig) - 0.5) * nl, 0.0, 1.0)
            cf_pert, _ = gen_fn(x_pert)
            if cf_pert is not None:
                m[f"R_IC (s={nl})"] = (torch.linalg.norm(cf_pert.view(-1) - cf.view(-1)).item()**2) / (torch.linalg.norm(cf.view(-1) - orig.view(-1)).item()**2 + 1e-6)
            else: m[f"R_IC (s={nl})"] = np.nan
        else: m[f"R_IC (s={nl})"] = np.nan

    # CHANGED: We now track the exact number of models in the set
    if ens and eps is not None:
        votes, members = 0, ens.get_rashomon_models(eps)
        for mm in members:
            mm.eval()
            with torch.no_grad(): votes += int(mm(cf).max(1)[1] == target)
        m["Rashomon Validity Ratio"] = votes / len(members)
        m["Rashomon Set Size"] = len(members)
    else:
        m["Rashomon Validity Ratio"] = np.nan
        m["Rashomon Set Size"] = np.nan

    return m

# ------------------------------------------------------------------------------------------
#For Experiments

def run_epsilon_experiment(epsilons, seeds, n_ensemble, n_samples_eval, epochs):
    results = {eps: defaultdict(lambda: defaultdict(list)) for eps in epsilons}
    saved_visuals = []

    for seed in seeds:
        set_seed(seed)
        print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")

        train_l, val_l, test_l, classes, lb, ub = get_pneumoniamnist_data()

        base_model = MCDropoutCNN().to(DEVICE)
        train_standard(base_model, train_l, epochs, "MC Dropout Base")

        ensemble = MCDropoutRashomonSet(base_model, n_samples=n_ensemble)
        ensemble.evaluate_all(val_l)

        #Print the number of models in each epsilon's Rashomon set to console
        print("\n  Rashomon Set Sizes for this seed:")
        for eps in epsilons:
            print(f"    eps={eps}: {len(ensemble.get_rashomon_models(eps))} models")
        print()

        surrogate = StandardCNN().to(DEVICE)
        train_standard(surrogate, train_l, epochs // 2, "CMV Surrogate")

        ae_gen = ImageAutoEncoder().to(DEVICE)
        train_ae(ae_gen, train_l, epochs // 2, "General AE")

        ae_cls = {}
        for c in range(2):
            idxs = [i for i, (_, y) in enumerate(train_l.dataset) if y.item() == c]
            if idxs:
                cl = DataLoader(train_l.dataset, batch_size=64, sampler=SubsetRandomSampler(idxs))
                ac = ImageAutoEncoder().to(DEVICE)
                train_ae(ac, cl, epochs // 2, f"Class {c} AE")
                ae_cls[c] = ac

        all_d = torch.cat([d for d, _ in train_l.dataset], dim=0).view(-1, 784)
        all_t = torch.tensor([t.item() for _, t in train_l.dataset])
        in_cls = {c: all_d[all_t == c] for c in range(2)}

        cvae_std = ImageCVAE().to(DEVICE)
        train_cvae_std(cvae_std, train_l, epochs, "CF-VAE")

        cvae_nerce = ImageCVAE().to(DEVICE)
        train_cvae_nerce(cvae_nerce, train_l, base_model, epochs, name="MM-CF-CVAE")

        cvae_eps = {}
        for eps in epsilons:
            cr = ImageCVAE().to(DEVICE)
            train_cvae_rashomon(cr, train_l, ensemble, eps, epochs)
            cvae_eps[eps] = cr

        print("\n--- Evaluating ---")
        base_model.train()
        processed = 0

        for x, y in test_l:
            if processed >= n_samples_eval: break
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)

            with torch.no_grad():
                pred = torch.stack([base_model(x) for _ in range(20)]).mean(0).max(1)[1].item()
            if pred != y.item(): continue

            processed += 1
            target = 1 - pred

            cf_std, t_std = gen_amortized(cvae_std, x, target)
            cf_ner, t_ner = gen_amortized(cvae_nerce, x, target)
            cf_bnn, t_bnn = gen_paper_bnn(base_model, x, target)
            cf_quc, t_quc = gen_quce(base_model, ae_gen, x, target)
            cf_grd, t_grd = gen_greedy(base_model, x, target)

            b_cfs = [
                ("CF-VAE", cf_std, t_std, lambda xp: gen_amortized(cvae_std, xp, target)),
                ("MM-CF-CVAE", cf_ner, t_ner, lambda xp: gen_amortized(cvae_nerce, xp, target)),
                ("Batten et al.", cf_bnn, t_bnn, lambda xp: gen_paper_bnn(base_model, xp, target)),
                ("QUCE", cf_quc, t_quc, lambda xp: gen_quce(base_model, ae_gen, xp, target)),
                ("Schut et al.", cf_grd, t_grd, lambda xp: gen_greedy(base_model, xp, target)),
            ]

            for eps in epsilons:
                for n, c, t, g in b_cfs:
                    if c is not None:
                        m = _collect_metrics(c, target, pred, x, base_model, surrogate, ae_cls, in_cls, t, ensemble, eps, g)
                        for k, v in m.items(): results[eps][n][k].append(v)

                cr = cvae_eps[eps]
                cf_r, t_r = gen_amortized(cr, x, target)
                if cf_r is not None:
                    m = _collect_metrics(cf_r, target, pred, x, base_model, surrogate, ae_cls, in_cls, t_r, ensemble, eps, lambda xp, _c=cr: gen_amortized(_c, xp, target))
                    for k, v in m.items(): results[eps][f"Rashomon (eps={eps})"][k].append(v)

            if len(saved_visuals) < 5:
                eps_to_plot = epsilons[len(epsilons)//2]
                cr_plot = cvae_eps[eps_to_plot]
                cf_r_plot, _ = gen_amortized(cr_plot, x, target)

                vis_dict = {
                    "orig": x.cpu(), "orig_cls": classes[pred], "tgt_cls": classes[target],
                    "cfs": {
                        "CF-VAE": cf_std.cpu(), "MM-CF-CVAE": cf_ner.cpu(),
                        "QUCE": cf_quc.cpu(), "Schut et al.": cf_grd.cpu(),
                        f"Rashomon (eps={eps_to_plot})": cf_r_plot.cpu() if cf_r_plot is not None else torch.zeros_like(x.cpu())
                    }
                }
                saved_visuals.append(vis_dict)

    return results, saved_visuals


#visualisation function
def plot_visuals(vis_data, out_dir=".", filename="pneumoniamnist_explanations.png"):
    if not vis_data: return
    methods = list(vis_data["cfs"].keys())
    fig, axes = plt.subplots(len(methods) + 1, 3, figsize=(10, 3 * (len(methods) + 1)), facecolor='white')

    orig_img = vis_data["orig"].numpy().reshape(28, 28)
    axes[0, 0].axis('off')
    axes[0, 1].imshow(orig_img, cmap='gray')
    axes[0, 1].set_title(f"Original Input\nPredicted: {vis_data['orig_cls']}")
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')

    for i, method in enumerate(methods):
        ax_row = i + 1
        cf_img = vis_data["cfs"][method].numpy().reshape(28, 28)
        diff_img = np.abs(cf_img - orig_img)

        axes[ax_row, 0].imshow(orig_img, cmap='gray')
        axes[ax_row, 0].set_title("Original")
        axes[ax_row, 0].axis('off')

        axes[ax_row, 1].imshow(cf_img, cmap='gray')
        axes[ax_row, 1].set_title(f"{method} CF\nTarget: {vis_data['tgt_cls']}")
        axes[ax_row, 1].axis('off')

        im = axes[ax_row, 2].imshow(diff_img, cmap='hot')
        axes[ax_row, 2].set_title("Difference Map")
        axes[ax_row, 2].axis('off')
        plt.colorbar(im, ax=axes[ax_row, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, filename), dpi=200, bbox_inches="tight")
    plt.close(fig)

#Execution over 5 random seeds & different deterministic epsilons
if __name__ == "__main__":
    EPSILONS     = [0.01, 0.05, 0.1, 0.2, 0.4, 0.8]
    SEEDS        = [11, 22, 33, 44, 55]
    N_ENSEMBLE   = 50
    N_EVAL       = 100
    EPOCHS       = 50
    OUT_DIR      = "./rashomon_results"

    os.makedirs(OUT_DIR, exist_ok=True)

    raw_results, vis_data_list = run_epsilon_experiment(
        epsilons=EPSILONS, seeds=SEEDS, n_ensemble=N_ENSEMBLE,
        n_samples_eval=N_EVAL, epochs=EPOCHS
    )

    #aggregate and save results
    rows = []
    for eps in EPSILONS:
        for meth, mets in raw_results[eps].items():
            row = {"Epsilon": eps, "Method": meth}
            for mk, mv in mets.items():
                fin = [v for v in mv if np.isfinite(v)]
                row[mk] = np.mean(fin) if fin else np.nan
                row[f"{mk}_std"] = np.std(fin) if fin else np.nan
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT_DIR, "results_pneumoniamnist.csv"), index=False)

    torch.save(vis_data_list, os.path.join(OUT_DIR, "all_vis_data.pt"))
    print(f"\nSaved raw visualization tensors to {os.path.join(OUT_DIR, 'all_vis_data.pt')}")

    for i, v_data in enumerate(vis_data_list):
        plot_visuals(v_data, OUT_DIR, filename=f"pneumoniamnist_explanations_instance_{i}.png")

    print("\nAll done! Results and visuals saved in", OUT_DIR)

# Visualise Explanation(s) Exploration

In [ ]:
"""
Counterfactual Playground - PneumoniaMNIST CNN - used claude/gemini to generate this for visualisation convenience.
===========================================================================
A streamlined script to generate and visualize image counterfactuals
without running the full metrics evaluation suite.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import os
import copy
import time

import medmnist
from medmnist import INFO

# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def get_bayesian_prediction_stats(model, x, target_class, n_samples=100):
    model.train() # Enable dropout
    x = x.to(DEVICE)
    batch_size = x.shape[0]
    x_rep = x.repeat(n_samples, 1, 1, 1)
    preds = model(x_rep).view(n_samples, batch_size, -1)
    log_probs = preds[:, :, target_class]
    mean_log_prob = log_probs.mean()
    return None, None, mean_log_prob

# ---------------------------------------------------------------------------
# 1. Data Handling
# ---------------------------------------------------------------------------
def get_pneumoniamnist_data():
    print("\n--- Loading PneumoniaMNIST dataset ---")
    info = INFO['pneumoniamnist']
    DataClass = getattr(medmnist, info['python_class'])
    transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = DataClass(split='train', transform=transform, download=True)
    test_dataset  = DataClass(split='test',  transform=transform, download=True)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=1,   shuffle=False)

    return train_loader, test_loader, ['Normal', 'Pneumonia']

# ---------------------------------------------------------------------------
# 2. Model Definitions
# ---------------------------------------------------------------------------
class MCDropoutCNN(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.drop1 = nn.Dropout2d(p=dropout_rate)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.drop2 = nn.Dropout2d(p=dropout_rate)
        self.fc1   = nn.Linear(320, 50)
        self.drop3 = nn.Dropout(p=dropout_rate)
        self.fc2   = nn.Linear(50, num_classes)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = self.drop1(x)
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = self.drop2(x)
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = self.drop3(x)
        return F.log_softmax(self.fc2(x), dim=-1)

class ImageAutoEncoder(nn.Module):
    def __init__(self, input_dim=784, latent_dim=20, hidden_dim=200):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid()
        )
    def forward(self, x):
        x_flat = x.view(x.size(0), -1)
        return self.decoder(self.encoder(x_flat)).view(x.size(0), 1, 28, 28)

class ImageCVAE(nn.Module):
    def __init__(self, num_classes=2, latent_dim=20, hidden_dim=400):
        super().__init__()
        self.label_embedding = nn.Embedding(num_classes, 50)
        self.enc_fc1  = nn.Linear(784 + 50, hidden_dim)
        self.enc_fc21 = nn.Linear(hidden_dim, latent_dim)
        self.enc_fc22 = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1  = nn.Linear(latent_dim + 50, hidden_dim)
        self.dec_fc2  = nn.Linear(hidden_dim, 784)

    def encode(self, x, y):
        y_emb = self.label_embedding(y)
        inputs = torch.cat([x.view(x.size(0), -1), y_emb], dim=1)
        h1 = F.relu(self.enc_fc1(inputs))
        return self.enc_fc21(h1), self.enc_fc22(h1)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z, y):
        inputs = torch.cat([z, self.label_embedding(y)], dim=1)
        h3 = F.relu(self.dec_fc1(inputs))
        return torch.sigmoid(self.dec_fc2(h3)).view(-1, 1, 28, 28)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar

# ---------------------------------------------------------------------------
# 3. Rashomon Ensemble for CNN
# ---------------------------------------------------------------------------
class FrozenMaskCNN(nn.Module):
    def __init__(self, base_model, p=0.2):
        super().__init__()
        self.base = copy.deepcopy(base_model)
        self.base.eval()
        self.mask1 = (torch.rand(1, 10, 1, 1) > p).float() / (1.0 - p)
        self.mask2 = (torch.rand(1, 20, 1, 1) > p).float() / (1.0 - p)
        self.mask3 = (torch.rand(1, 50) > p).float() / (1.0 - p)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.base.conv1(x), 2))
        x = x * self.mask1.to(x.device)
        x = F.relu(F.max_pool2d(self.base.conv2(x), 2))
        x = x * self.mask2.to(x.device)
        x = x.view(-1, 320)
        x = F.relu(self.base.fc1(x))
        x = x * self.mask3.to(x.device)
        return F.log_softmax(self.base.fc2(x), dim=-1)

class MCDropoutRashomonSet:
    def __init__(self, base_model, n_samples=50): # Default lowered for playground
        self.models = [FrozenMaskCNN(base_model).to(DEVICE) for _ in range(n_samples)]
        self.val_losses = np.random.rand(n_samples) # Mocked for speed in playground

    def get_rashomon_models(self, epsilon):
        L_star = self.val_losses.min()
        in_set = [m for m, l in zip(self.models, self.val_losses) if l <= L_star + epsilon]
        return in_set if in_set else [self.models[int(np.argmin(self.val_losses))]]

    def expected_log_prob_rashomon(self, x_prime, target_cf, epsilon):
        members = self.get_rashomon_models(epsilon)
        log_prob_sum = 0.0
        for m in members:
            lp = m(x_prime).gather(1, target_cf.view(-1, 1)).squeeze()
            log_prob_sum = log_prob_sum + lp
        return (log_prob_sum / len(members)).mean()

# ---------------------------------------------------------------------------
# 4. Training Helpers
# ---------------------------------------------------------------------------
def train_standard(model, loader, epochs, name):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    ce = nn.NLLLoss()
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            opt.zero_grad(); ce(model(x), y).backward(); opt.step()

def train_ae(model, loader, epochs, name):
    opt = optim.Adam(model.parameters(), lr=0.001)
    print(f"  Training {name}...")
    model.train()
    for _ in range(epochs):
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad(); F.mse_loss(model(x), x).backward(); opt.step()

def train_cvae_std(model, loader, epochs, name="CF-VAE"):
    opt = optim.Adam(model.parameters(), lr=0.001)
    print(f"  Training {name}...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            opt.zero_grad()
            recon, mu, logvar = model(x, y)
            bce = F.binary_cross_entropy(recon.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            (bce + kl).backward()
            opt.step()

def train_cvae_nerce(model, loader, base_model, epochs, name="MM-CF-CVAE"):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training {name}...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            target_cf = 1 - y
            mu, logvar = model.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z = model.reparameterize(mu, logvar)
            x_prime = model.decode(z, target_cf)

            base_model.train()
            M = 10
            exp_lp = sum(base_model(x_prime).gather(1, target_cf.view(-1, 1)).squeeze() for _ in range(M)) / M
            exp_lp = exp_lp.mean()

            prox = F.mse_loss(x_prime.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            loss = -(15.0 * exp_lp) + (1 * kl_div) + (1 * prox)
            opt.zero_grad(); loss.backward(); opt.step()

def train_cvae_rashomon(model, loader, ensemble, epsilon, epochs):
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"  Training Rashomon CVAE (eps={epsilon})...")
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)
            target_cf = 1 - y
            mu, logvar = model.encode(x, target_cf)
            kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
            z = model.reparameterize(mu, logvar)
            x_prime = model.decode(z, target_cf)

            exp_lp = ensemble.expected_log_prob_rashomon(x_prime, target_cf, epsilon)
            prox = F.mse_loss(x_prime.view(x.size(0), -1), x.view(x.size(0), -1), reduction='none').sum(dim=1).mean()
            loss = -(15.0 * exp_lp) + (1 * kl_div) + (1 * prox)
            opt.zero_grad(); loss.backward(); opt.step()

# ---------------------------------------------------------------------------
# 5. Counterfactual Generation
# ---------------------------------------------------------------------------
def gen_amortized(cvae, x_orig, target_class):
    cvae.eval()
    with torch.no_grad():
        yt = torch.tensor([target_class], device=DEVICE)
        mu, _ = cvae.encode(x_orig, yt)
        x_prime = cvae.decode(mu, yt)
    return x_prime.detach()

def gen_paper_bnn(model, x_orig, target_class):
    x_prime = x_orig.clone().detach().requires_grad_(True)
    opt = optim.Adam([x_prime], lr=0.1)
    for _ in range(200): # Lowered iteration count for quick visual check
        opt.zero_grad()
        _, _, mlp = get_bayesian_prediction_stats(model, x_prime, target_class)
        prox = torch.linalg.norm(x_prime.view(-1) - x_orig.view(-1))
        loss = -mlp + 0.01 * prox
        loss.backward()
        opt.step()
        with torch.no_grad(): x_prime.clamp_(0.0, 1.0)
    return x_prime.detach()

def gen_greedy(model, x_orig, target_class):
    x_prime = x_orig.clone().detach()
    altered = torch.zeros_like(x_prime, device=DEVICE)
    for _ in range(200): # Lowered for playground
        x_prime.requires_grad_(True)
        model.train()
        logits = model(x_prime.repeat(10, 1, 1, 1))
        loss = F.nll_loss(logits, torch.tensor([target_class]*10, device=DEVICE), reduction='none').mean()
        loss.backward()
        grad = x_prime.grad.clone().detach()
        x_prime = x_prime.detach()

        with torch.no_grad():
            gf = grad.view(-1)
            af = altered.view(-1)
            gf[af >= 20] = 0.0
            if gf.abs().max() == 0: break

            midx = torch.argmax(gf.abs())
            xp_flat = x_prime.view(-1)
            xp_flat[midx] -= 0.2 * gf[midx].sign()
            x_prime = xp_flat.view(1, 1, 28, 28).clamp(0.0, 1.0)
            af[midx] += 1
    return x_prime.detach()

def gen_quce(model, ae, x_orig, target_class):
    ae.eval()
    with torch.no_grad(): z0 = ae.encoder(x_orig.view(1, -1))
    z_prime = z0.clone().detach().requires_grad_(True)
    opt = optim.Adam([z_prime], lr=0.1)
    for _ in range(200): # Lowered for playground
        opt.zero_grad()
        x_prime = ae.decoder(z_prime).view(1, 1, 28, 28).clamp(0.0, 1.0)
        _, _, mlp = get_bayesian_prediction_stats(model, x_prime, target_class)
        recon_l = F.mse_loss(ae(x_prime), x_prime)
        prox_l = F.mse_loss(x_prime, x_orig)
        (5.0 * (-mlp) + 0.5 * recon_l + 0.5 * prox_l).backward()
        opt.step()
    return ae.decoder(z_prime).detach().view(1, 1, 28, 28).clamp(0.0, 1.0)

# ---------------------------------------------------------------------------
# 6. Plotting
# ---------------------------------------------------------------------------
def plot_explanations(orig, target_idx, class_names, cfs_dict):
    methods = list(cfs_dict.keys())
    num_cols = len(methods) + 1

    # Creates a wide, horizontal layout: 2 rows by (methods + 1) columns
    fig, axes = plt.subplots(2, num_cols, figsize=(2.8 * num_cols, 5), facecolor='white')

    orig_img = orig.cpu().numpy().reshape(28, 28)

    # --- Top Left: Original Image ---
    axes[0, 0].imshow(orig_img, cmap='gray')
    axes[0, 0].set_title(f"Original\n(Target: {class_names[target_idx]})", fontsize=14, fontweight='bold', pad=10)
    axes[0, 0].axis('off')

    # --- Bottom Left: Row Label ---
    axes[1, 0].axis('off')
    axes[1, 0].text(0.5, 0.5, 'Absolute\nDifference', horizontalalignment='center',
                    verticalalignment='center', fontsize=13, fontweight='bold')

    # Calculate global max difference so all heatmaps share the same color scale
    # This is crucial for scientifically fair visual comparisons in papers.
    diff_imgs = {}
    global_max_diff = 0.0
    for method in methods:
        cf_img = cfs_dict[method].cpu().numpy().reshape(28, 28)
        diff = np.abs(cf_img - orig_img)
        diff_imgs[method] = (cf_img, diff)
        if diff.max() > global_max_diff:
            global_max_diff = diff.max()

    # --- Populate the Grid ---
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    for i, method in enumerate(methods):
        col = i + 1
        cf_img, diff_img = diff_imgs[method]

        # Top Row: Counterfactual
        axes[0, col].imshow(cf_img, cmap='gray')
        # Use line breaks in titles if method names get too long
        title = method.replace(" (", "\n(")
        axes[0, col].set_title(title, fontsize=13, pad=10)
        axes[0, col].axis('off')

        # Bottom Row: Difference Map
        # Using 'magma' or 'hot' looks great in print; vmin/vmax ensures uniform scaling
        im = axes[1, col].imshow(diff_img, cmap='magma', vmin=0, vmax=global_max_diff)
        axes[1, col].axis('off')

        # Append a slim colorbar to each difference map
        divider = make_axes_locatable(axes[1, col])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = fig.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=9)

    # Tighten up the whitespace between subplots
    plt.subplots_adjust(wspace=0.05, hspace=0.1)

    # Save the figure cleanly if you want to drop it straight into LaTeX
    plt.savefig("cf_comparison.png", bbox_inches='tight', dpi=300)

    plt.show()
# ---------------------------------------------------------------------------
# 7. Main Playground Flow
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    EPOCHS = 50 # Fast training for quick visual iteration
    TEST_EPSILON = 0.1

    # 1. Load Data
    train_l, test_l, class_names = get_pneumoniamnist_data()

    # 2. Train Models
    base_model = MCDropoutCNN().to(DEVICE)
    train_standard(base_model, train_l, EPOCHS, "MC Dropout Base")

    ensemble = MCDropoutRashomonSet(base_model, n_samples=20)

    ae_gen = ImageAutoEncoder().to(DEVICE)
    train_ae(ae_gen, train_l, EPOCHS, "General AE (for QUCE)")

    cvae_std = ImageCVAE().to(DEVICE)
    train_cvae_std(cvae_std, train_l, EPOCHS, "CF-VAE")

    cvae_nerce = ImageCVAE().to(DEVICE)
    train_cvae_nerce(cvae_nerce, train_l, base_model, EPOCHS, "MM-CF-CVAE (NERCE)")

    cvae_rashomon = ImageCVAE().to(DEVICE)
    train_cvae_rashomon(cvae_rashomon, train_l, ensemble, TEST_EPSILON, EPOCHS)

    # 3. Fetch an image to explain
    print("\n--- Generating Explanations ---")
    base_model.eval()

    for x, y in test_l:
        x, y = x.to(DEVICE), y.squeeze(1).long().to(DEVICE)

        # Make sure the base model actually predicts this correctly first
        with torch.no_grad():
            pred = torch.stack([base_model(x) for _ in range(20)]).mean(0).max(1)[1].item()

        if pred == y.item():
            target_class = 1 - pred
            print(f"Found suitable image. Predicted: {class_names[pred]}, Target CF: {class_names[target_class]}")

            # Generate CFs
            cfs = {
                #"CF-VAE": gen_amortized(cvae_std, x, target_class),
                "AVCG": gen_amortized(cvae_nerce, x, target_class),
                #"QUCE": gen_quce(base_model, ae_gen, x, target_class),
                "Batten et al.": gen_paper_bnn(base_model, x, target_class),
                "Schut et al.": gen_greedy(base_model, x, target_class),
                f"Rashomon AVCG (eps={TEST_EPSILON})": gen_amortized(cvae_rashomon, x, target_class)
            }

            # 4. Plot
            plot_explanations(x, target_class, class_names, cfs)
            break # Stop after the first image